In [ ]:
%run ./globalvariables

In [ ]:
from pathlib import Path

import pandas as pd

In [ ]:
spark.sql(f"CREATE CATALOG IF NOT EXISTS {CATALOG}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {BRONZE_TABLE}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {SILVER_TABLE}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {GOLD_TABLE}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {INFRA_TABLE}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {ML_TABLE}")

In [ ]:
spark.sql(f"CREATE VOLUME IF NOT EXISTS {INFRA_TABLE}.config")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {BRONZE_TABLE}.schema")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {BRONZE_TABLE}.landing")

In [ ]:
# Seed config once
ORIGIN_SEED = """id,type,origin,endpoint,secret_scope,client,database_name,port
1,CKAN,datos_madrid,https://datos.madrid.es/api/3/action,,,,
"""

DATASET_SEED = """id,dataset_origin,origin,dataset_destino,load_enabled,file_type,partition,select_by
1,201410-0-calidad-aire-diario,1,aire,1,csv,year,description
2,212629-0-estaciones-control-aire,1,estaciones_aire,1,csv,snapshot,format
3,208627-0-transporte-ptomedida-historico,1,trafico,1,zip,month,description
4,202468-0-intensidad-trafico,1,trafico_puntos_medida,1,csv,month,description
5,300497-0-distritos-municipales-madrid,1,distritos,1,shapefile,snapshot,format
"""

for path, content in [(ORIGIN_PATH, ORIGIN_SEED), (DATASET_PATH, DATASET_SEED)]:
    if Path(path).exists():
        print(f"already there, untouched: {path}")
    else:
        Path(path).parent.mkdir(parents=True, exist_ok=True)
        Path(path).write_text(content)
        print(f"seeded: {path}")

In [ ]:
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {INFRA_TABLE}.error_logs (
        timestamp STRING,
        notebook_name STRING,
        error_message STRING,
        status STRING,
        run_id STRING
    )
    USING DELTA
""")

In [ ]:
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {INFRA_TABLE}.ingestion_log (
        timestamp STRING,
        notebook_name STRING,
        dataset_destino STRING,
        partition_key STRING,
        rows_written LONG,
        status STRING,
        run_id STRING
    )
    USING DELTA
""")

In [ ]:
df_origins = pd.read_csv(ORIGIN_PATH)
(spark.createDataFrame(df_origins.astype(str))
    .write.mode("overwrite").option("mergeSchema", "true")
    .saveAsTable(f"{INFRA_TABLE}.origins"))
print(f"{INFRA_TABLE}.origins: {len(df_origins)} rows")

In [ ]:
df_datasets = pd.read_csv(DATASET_PATH)
(spark.createDataFrame(df_datasets.astype(str))
    .write.mode("overwrite").option("mergeSchema", "true")
    .saveAsTable(f"{INFRA_TABLE}.datasets"))
print(f"{INFRA_TABLE}.datasets: {len(df_datasets)} rows")

In [ ]:
spark.sql(f"SHOW TABLES IN {INFRA_TABLE}").show(truncate=False)
spark.sql(f"SHOW VOLUMES IN {INFRA_TABLE}").show(truncate=False)
spark.sql(f"SHOW VOLUMES IN {BRONZE_TABLE}").show(truncate=False)
spark.sql(f"SELECT * FROM {INFRA_TABLE}.datasets").show(truncate=False)